# 02 — Deep-Dive Investigation: "ARPDAU is Falling. Fix It."

**The Problem Presented by Leadership:**  
> *"Comparing the first 30 days of the window (January 1–30) to the last 30 days (April 1–30), overall game ARPDAU is down roughly 16%. Leadership wants to know what broke in monetization and how to fix it."*

**The Mystery:**  
When we examine ARPDAU segmented by **`country_tier`**, every single tier is **flat or improving**!  
How can a business be earning *more* per customer in every country group, while the total company average *crashes by 16%*?  

This notebook provides the complete mathematical and business forensic analysis using **Shift-Share Decomposition** and uncovers the true root cause.

## Setup & Loading Views

We load our helpers and compile our cleaning views and decomposition SQL script.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from python.helpers import q, run_sql_file

# Compile base cleaning views and deep dive SQL views
run_sql_file('00_clean_views.sql')
run_sql_file('02_arpdau_decomp.sql')

print('SQL views compiled successfully.')

---
## 1. Confirming the Paradox: Overall ARPDAU vs Tier ARPDAU

### What are we trying to find out?
We compare the first 30 days of the window (**Period 1: Jan 1 – Jan 30, 2026**) against the last 30 days (**Period 2: Apr 1 – Apr 30, 2026**) across two levels:
1. **Game-level aggregate:** Did overall ARPDAU drop by ~16%?
2. **Tier-level breakdown:** Did ARPDAU rise or fall within individual country tiers?

### Why does it matter?
If monetization truly broke (e.g., bad store pricing, broken IAP button, unappealing item packs), we would expect revenue per player to fall within individual countries. If it didn't, the problem is not in monetization mechanics at all.

In [ ]:
# Query the tier-level performance view
df_tier_metrics = q('SELECT * FROM arpdau_period_by_tier')
print('Tier-Level Performance by Period:\n')
print(df_tier_metrics.to_string(index=False))

# Compute overall ARPDAU for Period 1 vs Period 2
df_overall = q('''
    SELECT 
        period,
        SUM(total_dau) AS total_dau,
        ROUND(SUM(total_revenue), 2) AS total_revenue,
        ROUND(SUM(total_revenue) / SUM(total_dau), 4) AS overall_arpdau
    FROM arpdau_period_by_tier
    GROUP BY period
    ORDER BY period
''')

p1_arpdau = df_overall.loc[df_overall['period'].str.startswith('Period 1'), 'overall_arpdau'].values[0]
p2_arpdau = df_overall.loc[df_overall['period'].str.startswith('Period 2'), 'overall_arpdau'].values[0]
pct_change = round((p2_arpdau - p1_arpdau) * 100.0 / p1_arpdau, 2)

print('\nOverall Game Performance:')
print(df_overall.to_string(index=False))
print(f'\nOverall ARPDAU Change: ${p1_arpdau:.4f} → ${p2_arpdau:.4f} ({pct_change:+}%). Confirming ~16% drop!')

### What this tells us

The data confirms the exact paradox:
- **Overall ARPDAU dropped by 16.3%** (from **\$0.3567** down to **\$0.2985**).
- Yet **every single country tier improved**:
  - **Tier 1:** \$0.5235 → \$0.5602 (**+7.0% UP**)
  - **Tier 2:** \$0.3754 → \$0.4073 (**+8.5% UP**)
  - **Tier 3:** \$0.2339 → \$0.2654 (**+13.5% UP**)
  - **Tier 4:** \$0.1552 → \$0.1582 (**+1.9% UP**)

This is classic **Simpson's Paradox**: a trend appears in every subgroup, but completely reverses when aggregated together.

---
## 2. Share of DAU by Tier: Unveiling the Mix Shift

### What are we trying to find out?
How did the proportion of active players (DAU) coming from each country tier change between Period 1 and Period 2?

### Why does it matter?
An overall average is a weighted average:  
$$\text{Overall ARPDAU} = \sum (\text{Tier DAU Share} \times \text{Tier ARPDAU})$$  
If the weights shift dramatically toward lower-monetizing groups, the overall average will drop even if individual performance rises.

In [ ]:
# Pivot the tier data to clearly show DAU share shifts
df_pivot_share = q('''
    SELECT 
        country_tier,
        ROUND(MAX(CASE WHEN period LIKE 'Period 1%' THEN dau_share END) * 100, 2) AS p1_dau_share_pct,
        ROUND(MAX(CASE WHEN period LIKE 'Period 2%' THEN dau_share END) * 100, 2) AS p2_dau_share_pct,
        ROUND((MAX(CASE WHEN period LIKE 'Period 2%' THEN dau_share END) - 
               MAX(CASE WHEN period LIKE 'Period 1%' THEN dau_share END)) * 100, 2) AS share_change_pct_points,
        ROUND(MAX(CASE WHEN period LIKE 'Period 1%' THEN total_dau END), 0) AS p1_total_dau,
        ROUND(MAX(CASE WHEN period LIKE 'Period 2%' THEN total_dau END), 0) AS p2_total_dau
    FROM arpdau_period_by_tier
    GROUP BY country_tier
    ORDER BY country_tier
''')

print('DAU Share Shift by Country Tier:')
print(df_pivot_share.to_string(index=False))

### What this tells us

A massive compositional transformation took place:
- In January (Period 1), **Tier 1 made up 34.5%** of all active players, while Tier 4 was only **17.7%**.
- By April (Period 2), **Tier 4 surged to 46.0% of all active players** (nearly half of the entire game!), while Tier 1 shrank to **20.4%**.
- Tier 4 DAU exploded nearly **9-fold** (from 14,543 to 130,045 active player-days).
- Because Tier 4 players generate ~\$0.16/day compared to Tier 1's ~\$0.56/day, flooding the game with Tier 4 players dragged down the overall average.

---
## 3. Shift-Share Decomposition

### 💡 Concept Box: Shift-Share Decomposition
Shift-share decomposition is a rigorous mathematical technique used by economists to answer:  
*"How much of our metric change is due to real performance changes vs. population composition changes?"*

$$\Delta R = \underbrace{\sum w_{i,1} \cdot \Delta r_i}_{\text{Within-Tier Effect}} + \underbrace{\sum \Delta w_i \cdot r_{i,1}}_{\text{Mix Effect (Between)}} + \underbrace{\sum \Delta w_i \cdot \Delta r_i}_{\text{Interaction Effect}}$$

1. **Within-Tier Effect:** What would have happened if player mix stayed identical to January, but each tier's monetization changed as observed? (Measures game design & monetization health).
2. **Mix Effect:** What would have happened if monetization rates stayed frozen at January levels, but the player mix shifted as observed? (Measures marketing acquisition impact).
3. **Interaction Effect:** The cross-product term of moving both weights and rates simultaneously.

In [ ]:
# Execute the shift-share decomposition view
df_decomp = q('SELECT * FROM arpdau_shift_share_decomp')
print('Shift-Share Decomposition by Tier:\n')
print(df_decomp.to_string(index=False))

print('\n' + '='*55 + '\n')
df_decomp_summary = q('SELECT * FROM arpdau_shift_share_summary')
print('Shift-Share Decomposition Summary (% of Total ARPDAU Change):\n')
print(df_decomp_summary.to_string(index=False))

### What this tells us

The decomposition proves the case conclusively:
- **Total ARPDAU Change:** **-\$0.0582** (-16.3%)
- **Within-Tier Effect: +\$0.0284 (-48.8% of the change)**  
  If player composition had not changed, ARPDAU would have **risen by nearly \$0.03 (+8.0%)**! The monetization team actually did an outstanding job.
- **Mix Effect: -\$0.0778 (+133.6% of the change)**  
  The shift in player country mix is responsible for **more than 100% of the entire ARPDAU collapse**!
- **Interaction Effect: -\$0.0088 (+15.2% of the change)**  
  The residual joint adjustment.

---
## 4. What Caused the Mix Shift? (Channel & Country Drivers)

### What are we trying to find out?
Where did the 115,000+ new Tier 4 active player-days come from? Was it organic virality, paid social campaigns, or paid video ads? In which specific countries?

In [ ]:
# Query channel growth across tiers
df_channel_drivers = q('SELECT * FROM mix_shift_channel_drivers')
print('Install Growth Ratio (P1 vs P2) by Tier and Channel:\n')
print(df_channel_drivers.to_string(index=False))

# Pinpoint the exact countries driving Tier 4 expansion
df_top_tier4_countries = q('''
    SELECT 
        country,
        acquisition_channel,
        COUNT(CASE WHEN install_date BETWEEN '2026-01-01' AND '2026-01-30' THEN 1 END) AS p1_installs,
        COUNT(CASE WHEN install_date BETWEEN '2026-04-01' AND '2026-04-30' THEN 1 END) AS p2_installs,
        ROUND(COUNT(CASE WHEN install_date BETWEEN '2026-04-01' AND '2026-04-30' THEN 1 END) * 1.0 / 
              NULLIF(COUNT(CASE WHEN install_date BETWEEN '2026-01-01' AND '2026-01-30' THEN 1 END), 0), 1) AS growth_factor
    FROM profile_clean
    WHERE country_tier = 4
    GROUP BY country, acquisition_channel
    ORDER BY p2_installs DESC
    LIMIT 8
''')

print('\nTop Tier 4 Countries Fueling the Surge:')
print(df_top_tier4_countries.to_string(index=False))

### What this tells us

The smoking gun is found:
1. **Organic and Paid Social were completely flat:** Organic installs grew by only 1.01×–1.06× across all tiers, and paid social grew by 1.02×–1.05×.
2. **Tier 4 `paid_video` exploded 57-fold:** In January, Tier 4 paid video generated only **266 installs**. In April, it generated **15,272 installs**!
3. **Driven by 5 specific emerging markets:** The User Acquisition (UA) team poured budget into low-cost video networks in **Indonesia, Egypt, Philippines, India, and Vietnam** (~3,000 installs each in April, up from ~50 each in January).
4. **Why did marketing do this?** Ad networks offer extraordinarily cheap cost-per-install (CPI) in Tier 4 countries. The UA team likely chased install volume targets without accounting for player purchasing power.

---
## 5. Explaining Simpson's Paradox to Leadership: The Coffee & Tea Shop Analogy

To explain this to the Product Director and executive team without drowning in statistical equations, use this analogy:

> ### ☕ The Café Analogy
> Imagine our studio runs a café that sells two drinks: **Specialty Coffee** for **\$5.00** and **Basic Green Tea** for **\$1.00**.
>
> - **In January**, the café serves **80 coffee drinkers** and **20 tea drinkers** (100 customers total).  
>   $$\text{Average spend per customer} = \frac{(80 \times \$5.00) + (20 \times \$1.00)}{100} = \frac{\$420}{100} = \mathbf{\$4.20}$$
>
> - **In April**, the café improves its menu and raises prices slightly: Coffee is now **\$5.50** (+10%) and Tea is now **\$1.20** (+20%). *Every single beverage is earning more money than ever before!*
>
> - **However, in April**, the marketing team hands out flyers at a nearby school, and a tour bus of students arrives: the café serves **80 coffee drinkers** and **920 tea drinkers** (1,000 customers total).  
>   $$\text{Total Revenue} = (80 \times \$5.50) + (920 \times \$1.20) = \$440 + \$1,104 = \$1,544$$  
>   $$\text{Average spend per customer} = \frac{\$1,544}{1,000} = \mathbf{\$1.54}$$
>
> **The Executive Panic:** The manager looks at the dashboard and panics: *"Our average customer spend crashed by 63% (from \$4.20 to \$1.54)! Did our coffee taste bad? Did our cash register break?"*  
> **The Truth:** Nothing broke. Revenue actually grew from \$420 to \$1,544! The average spend per customer fell solely because 92% of the crowd bought tea.
>
> In our mobile game, **Tier 1 is Coffee (\$0.56/day)** and **Tier 4 is Tea (\$0.16/day)**. The marketing team brought in a tour bus of Tier 4 players.

---
## 6. Business Recommendations & The Missing Finance Input

### What Broke?
**Nothing broke in monetization.** Game economy, store pricing, and player willingness to pay are stronger than ever. The drop in ARPDAU is entirely an artifact of a dramatic user acquisition mix shift toward low-monetizing developing countries.

### What Input is Missing from the Data?
This dataset contains **player telemetry only** (in-game behavior and gross IAP revenue), but **zero marketing finance data**. Specifically, we do not have:
- **Cost Per Install (CPI)** by country tier and acquisition channel.
- **Marketing Spend Budgets** allocated per campaign.

Without CPI, we cannot determine whether the Tier 4 paid video campaign was a commercial success or an expensive blunder.

### The Strategic Decision Framework (Phrased for Leadership)
An average Tier 4 player has an ARPDAU of **\$0.1582** per active day and averages roughly 4–5 active days in their first month, yielding approximately **\$0.65 to \$0.80 in 30-day revenue**.

> **Decision Rule:**  
> *"Acquiring Tier 4 players via paid video is commercially sound **if and only if** the blended Cost Per Install (CPI) in Indonesia, India, Vietnam, Egypt, and the Philippines is **under \$0.40** (guaranteeing positive return on ad spend). If the UA team is paying \$0.75+ per install, the studio is losing money on every single download, and that ad budget should immediately be redirected back to Tier 1 and Tier 2 channels."*

### Action Items for the Product Director:
1. **Do NOT alter store pricing or increase paywalls:** The product is healthy; tampering with core economy parameters to "fix ARPDAU" would risk alienating high-value players.
2. **Audit UA spend with Finance:** Compare actual CPI by country against 30-day realized player value.
3. **Establish Tier-Specific KPIs:** Stop evaluating game health on blended aggregate ARPDAU. Track **Tier 1 ARPDAU** and **Tier 4 ARPDAU** on separate dashboard tiles to avoid being misled by Simpson's Paradox in the future.

---
## 7. Executive Visualizations

Below are the two charts generated for the Product Director's presentation, saved to `outputs/charts/`.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 11))

img1 = mpimg.imread('../outputs/charts/02_simpsons_paradox_key.png')
ax1.imshow(img1)
ax1.axis('off')
ax1.set_title('Figure 1: Key Executive Chart — Simpson\'s Paradox in ARPDAU', fontsize=13, fontweight='bold', pad=10)

img2 = mpimg.imread('../outputs/charts/02_dau_mix_shift_supporting.png')
ax2.imshow(img2)
ax2.axis('off')
ax2.set_title('Figure 2: Supporting Chart — The DAU Compositional Shift', fontsize=13, fontweight='bold', pad=10)

plt.tight_layout()
plt.show()

---
## Stage 2 Summary

All questions answered, decomposition computed, analogy framed, decision rule formulated, and charts saved.

**Ready for Stage 3 when you say so.**